In [23]:
from src.ingest_lyrics import setup_postgres, check_tables
from src.embeddings.embedder import Embedder
from src.search_pgvector import search_sections, aggregate_song_results

embedder = Embedder()
conn = setup_postgres()
check_tables(conn)

query = "I want to cheer myself up after a stressful day at work."
top_k_sections = search_sections(query, embedder, conn)
top_k_songs = aggregate_song_results(top_k_sections)

Connecting to PostgreSQL...
Connected!
pgvector extension ready!
4084 songs and 41681 lyrics chunk documents in database.


In [24]:
top_k_sections

[{'document_id': 17304,
  'song_id': 1701,
  'title': 'I Only Miss You',
  'performer': 'Megan Moroney & Ed Sheeran',
  'wks_on_chart': 1,
  'peak_pos': 67,
  'section_id': 1,
  'section': "I thought today would be the day\nThat I'd finally finish sober\nI'd cook a meal and go to bed\nWithout sadness in my chest\nWake up with no hangover",
  'num_lines': 5,
  'similarity': 0.5256215020643779},
 {'document_id': 31720,
  'song_id': 3098,
  'title': 'Silent Hill',
  'performer': 'Kendrick Lamar & Kodak Black',
  'wks_on_chart': 11,
  'peak_pos': 7,
  'section_id': 11,
  'section': "Head up, chest out\nSilence, I'm stressed out\nShh, be quiet, I'm stressed out\nStressed out, stressed out, stressed out",
  'num_lines': 4,
  'similarity': 0.5189794588697288},
 {'document_id': 31297,
  'song_id': 3063,
  'title': 'She Knows This',
  'performer': 'Kid Cudi',
  'wks_on_chart': 1,
  'peak_pos': 49,
  'section_id': 9,
  'section': "See I can't be stressin' (no, no, I can't stress)\nI just need my

In [25]:
top_k_songs

[{'song_id': 1701,
  'title': 'I Only Miss You',
  'performer': 'Megan Moroney & Ed Sheeran',
  'wks_on_chart': 1,
  'peak_pos': 67,
  'best_similarity': 0.5256215020643779,
  'num_matches': 1,
  'matched_sections': [{'section_id': 1,
    'section': "I thought today would be the day\nThat I'd finally finish sober\nI'd cook a meal and go to bed\nWithout sadness in my chest\nWake up with no hangover",
    'num_lines': 5,
    'similarity': 0.5256215020643779}]},
 {'song_id': 3098,
  'title': 'Silent Hill',
  'performer': 'Kendrick Lamar & Kodak Black',
  'wks_on_chart': 11,
  'peak_pos': 7,
  'best_similarity': 0.5189794588697288,
  'num_matches': 1,
  'matched_sections': [{'section_id': 11,
    'section': "Head up, chest out\nSilence, I'm stressed out\nShh, be quiet, I'm stressed out\nStressed out, stressed out, stressed out",
    'num_lines': 4,
    'similarity': 0.5189794588697288}]},
 {'song_id': 3063,
  'title': 'She Knows This',
  'performer': 'Kid Cudi',
  'wks_on_chart': 1,
  'pea

In [3]:
from src.rerank_songs import rerank_songs
new_top_k_songs = rerank_songs(top_k_songs)
new_top_k_songs

/Users/chs/Documents/LLM-zoomcamp/Capstone/LyricLens/src/rerank_songs.py:26: RuntimeWarning: invalid value encountered in divide
  return min_goal + (nums - min_num) / range_num * range_goal


[{'song_id': 1701,
  'title': 'I Only Miss You',
  'performer': 'Megan Moroney & Ed Sheeran',
  'wks_on_chart': 1,
  'peak_pos': 67,
  'best_similarity': 0.5256215020643779,
  'num_matches': 2,
  'matched_sections': [{'section_id': 1,
    'section': "I thought today would be the day\nThat I'd finally finish sober\nI'd cook a meal and go to bed\nWithout sadness in my chest\nWake up with no hangover",
    'num_lines': 5,
    'similarity': 0.5256215020643779}],
  'overall_score': nan},
 {'song_id': 3098,
  'title': 'Silent Hill',
  'performer': 'Kendrick Lamar & Kodak Black',
  'wks_on_chart': 11,
  'peak_pos': 7,
  'best_similarity': 0.5189794588697288,
  'num_matches': 2,
  'matched_sections': [{'section_id': 11,
    'section': "Head up, chest out\nSilence, I'm stressed out\nShh, be quiet, I'm stressed out\nStressed out, stressed out, stressed out",
    'num_lines': 4,
    'similarity': 0.5189794588697288}],
  'overall_score': nan},
 {'song_id': 3063,
  'title': 'She Knows This',
  'per

In [4]:
from src.rerank_songs import weighted_total_score
song_scores = weighted_total_score(top_k_songs)
song_scores

/Users/chs/Documents/LLM-zoomcamp/Capstone/LyricLens/src/rerank_songs.py:26: RuntimeWarning: invalid value encountered in divide
  return min_goal + (nums - min_num) / range_num * range_goal


array([nan, nan, nan, nan, nan])

In [5]:
from src.rerank_songs import minmax_linear_map
minmax_linear_map([1,2,3], 0, 1)

array([0. , 0.5, 1. ])

In [6]:
from src.rerank_songs import wks_on_chart_score
wks_on_chart_score([song["wks_on_chart"] for song in top_k_songs])

array([0.07692308, 0.84615385, 0.07692308, 0.15384615, 1.        ])

In [7]:
from src.rerank_songs import peak_pos_score
peak_pos_score([song["peak_pos"] for song in top_k_songs])

array([0.27173913, 0.78125   , 0.33783784, 0.36764706, 0.52083333])

In [8]:
from src.rerank_songs import best_sim_score
best_sim_score([song["best_similarity"] for song in top_k_songs])    

array([1.        , 0.94561048, 0.68828303, 0.59418784, 0.45963179])

In [9]:
from src.rerank_songs import num_matches_score
num_matches_score([song["num_matches"] for song in top_k_songs])

/Users/chs/Documents/LLM-zoomcamp/Capstone/LyricLens/src/rerank_songs.py:26: RuntimeWarning: invalid value encountered in divide
  return min_goal + (nums - min_num) / range_num * range_goal


array([nan, nan, nan, nan, nan])

In [10]:
[song["num_matches"] for song in top_k_songs]

[2, 2, 2, 2, 2]

In [1]:
# autoreload every imported modules if their source files have changed.
%load_ext autoreload
%autoreload 2

In [16]:
num_matches_score([song["num_matches"] for song in top_k_songs])

array([nan, nan, nan, nan, nan])

In [17]:
[song["num_matches"] for song in top_k_songs]

[2, 2, 2, 2, 2]

In [18]:
num_matches_score([1,2])

array([0.05, 1.  ])

In [19]:
num_matches_score([1,1])

array([nan, nan])

---

In [26]:
from src.rerank_songs import rerank_songs
new_top_k_songs = rerank_songs(top_k_songs)
new_top_k_songs

[{'song_id': 3098,
  'title': 'Silent Hill',
  'performer': 'Kendrick Lamar & Kodak Black',
  'wks_on_chart': 11,
  'peak_pos': 7,
  'best_similarity': 0.5189794588697288,
  'num_matches': 1,
  'matched_sections': [{'section_id': 11,
    'section': "Head up, chest out\nSilence, I'm stressed out\nShh, be quiet, I'm stressed out\nStressed out, stressed out, stressed out",
    'num_lines': 4,
    'similarity': 0.5189794588697288}],
  'overall_score': 0.926230720146022},
 {'song_id': 1701,
  'title': 'I Only Miss You',
  'performer': 'Megan Moroney & Ed Sheeran',
  'wks_on_chart': 1,
  'peak_pos': 67,
  'best_similarity': 0.5256215020643779,
  'num_matches': 1,
  'matched_sections': [{'section_id': 1,
    'section': "I thought today would be the day\nThat I'd finally finish sober\nI'd cook a meal and go to bed\nWithout sadness in my chest\nWake up with no hangover",
    'num_lines': 5,
    'similarity': 0.5256215020643779}],
  'overall_score': 0.7935827759197325},
 {'song_id': 3896,
  'tit

In [27]:
new_top_k_songs[0]

{'song_id': 3098,
 'title': 'Silent Hill',
 'performer': 'Kendrick Lamar & Kodak Black',
 'wks_on_chart': 11,
 'peak_pos': 7,
 'best_similarity': 0.5189794588697288,
 'num_matches': 1,
 'matched_sections': [{'section_id': 11,
   'section': "Head up, chest out\nSilence, I'm stressed out\nShh, be quiet, I'm stressed out\nStressed out, stressed out, stressed out",
   'num_lines': 4,
   'similarity': 0.5189794588697288}],
 'overall_score': 0.926230720146022}

In [28]:
new_top_k_songs[1]

{'song_id': 1701,
 'title': 'I Only Miss You',
 'performer': 'Megan Moroney & Ed Sheeran',
 'wks_on_chart': 1,
 'peak_pos': 67,
 'best_similarity': 0.5256215020643779,
 'num_matches': 1,
 'matched_sections': [{'section_id': 1,
   'section': "I thought today would be the day\nThat I'd finally finish sober\nI'd cook a meal and go to bed\nWithout sadness in my chest\nWake up with no hangover",
   'num_lines': 5,
   'similarity': 0.5256215020643779}],
 'overall_score': 0.7935827759197325}

In [29]:
new_top_k_songs[2]

{'song_id': 3896,
 'title': 'Which One',
 'performer': 'Drake & Central Cee',
 'wks_on_chart': 13,
 'peak_pos': 23,
 'best_similarity': 0.4596317869534574,
 'num_matches': 1,
 'matched_sections': [{'section_id': 9,
   'section': "Then work, work, work, work, work\nYeah, work, work, work, work, work\nYour last man broke your heart and it hurts\nYou could cry out ya eye and curse\nYou want diamond watch, you want purse\nYou don't need swimsuit, take off your shirt",
   'num_lines': 6,
   'similarity': 0.4596317869534574}],
 'overall_score': 0.6699200601433954}

In [30]:
new_top_k_songs[3]

{'song_id': 3063,
 'title': 'She Knows This',
 'performer': 'Kid Cudi',
 'wks_on_chart': 1,
 'peak_pos': 49,
 'best_similarity': 0.487554654849929,
 'num_matches': 1,
 'matched_sections': [{'section_id': 9,
   'section': "See I can't be stressin' (no, no, I can't stress)\nI just need my medicine (yeah baby, I need it)\nBaby, come and learn these lessons (come, baby, and see)\nBeen around and around again (boom-boom, boom-boom)",
   'num_lines': 4,
   'similarity': 0.487554654849929}],
 'overall_score': 0.6459866281324393}

In [31]:
new_top_k_songs[4]

{'song_id': 1849,
 'title': 'Just Like Magic',
 'performer': 'Ariana Grande',
 'wks_on_chart': 2,
 'peak_pos': 43,
 'best_similarity': 0.4760637595615539,
 'num_matches': 1,
 'matched_sections': [{'section_id': 2,
   'section': "Wake up in my bed, I just wanna have a good day\nThink it in my head, then it happens how it should, ay\nTwelve o'clock, I got a team meeting, then a meditation at like 1:30\nThen I ride to the studio, listening to some - I wrote",
   'num_lines': 4,
   'similarity': 0.4760637595615539}],
 'overall_score': 0.6122805720995149}